# Sistema Híbrido de Contagem de Pessoas — YOLOv11 + CSRNet

Arquivo **único** que integra dois modelos e um **roteador** que escolhe automaticamente qual usar:

- **Cena esparsa** (sala, rua, evento) → **YOLOv11** detecta cada pessoa e conta as caixas.
- **Multidão densa** (estádio, show, manifestação) → **CSRNet** estima um mapa de densidade e conta pela integral.
- **Roteador** → roda o YOLO primeiro; se a cena for densa demais para ele, escala para o CSRNet.

> **Base do projeto:** o pipeline de detecção do YOLO (carregar modelo → inferir → filtrar "person" →
> contar → exibir anotada) vem do notebook inicial do colega. Esse núcleo permanece vivo aqui, no Modelo A —
> apenas corrigido e estendido. Os Modelos B e o roteador foram construídos por cima dessa fundação.

## Roteiro
0. Setup → 1. **Modelo A (YOLO)** → 2. **Modelo B (CSRNet): dados, treino, avaliação** →
3. **Roteador híbrido** → 4. Demonstração integrada.

**GPU:** *Ambiente de execução → Alterar o tipo → T4*. O treino do CSRNet salva checkpoint a cada
época no Drive e **retoma de onde parou** se a sessão cair.


## 0. Setup

In [ ]:
# Instalação e imports
!pip install -q ultralytics

import os, glob, csv, time, random, json
import numpy as np
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from ultralytics import YOLO
import scipy.io as sio
from scipy.ndimage import gaussian_filter
import cv2
import matplotlib.pyplot as plt
from PIL import Image

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Torch:", torch.__version__, "| Dispositivo:",
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU (ative a T4!)")


In [ ]:
# === Parâmetros do projeto ===
# -- CSRNet (modelo denso) --
PART       = "A"      # "A" = densa (estádio/show) | "B" = esparsa (rua/sala)
EPOCHS     = 50       # comece com 10-20 p/ validar o fluxo; depois aumente
LR         = 1e-5     # Adam
SIGMA      = 15       # desvio da gaussiana do ground truth (kernel fixo)
MAX_TRAIN  = None     # ex.: 40 p/ teste-relâmpago; None = dataset inteiro
SEED       = 42

# -- Roteador híbrido (limiares calibráveis) --
LIMIAR_CONTAGEM     = 50     # acima disso, considera cena densa
LIMIAR_CONTAGEM_MIN = 20     # a partir daqui, o tamanho das caixas pode indicar densa
LIMIAR_TAMANHO      = 0.01   # caixa média < 1% da área da imagem = cabeças pequenas = denso

# -- Caminhos (Drive) --
DRIVE_PROJ = "/content/drive/MyDrive/contagem_multidao"

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)


In [ ]:
# Montar o Drive (checkpoints e cache do dataset)
from google.colab import drive
drive.mount('/content/drive')
CKPT_DIR = os.path.join(DRIVE_PROJ, f"csrnet_part{PART}")
os.makedirs(CKPT_DIR, exist_ok=True)
print("Projeto:", DRIVE_PROJ)


## 1. Modelo A — YOLOv11 (cenas esparsas)

Núcleo herdado do notebook do colega: carrega o YOLO, detecta pessoas, filtra a classe "person",
conta e exibe a imagem anotada. **Correções aplicadas** sobre o original:
- **Bug de cor (BGR→RGB):** `plot()` retorna BGR; convertemos para RGB antes de exibir.
- **Contagem por nome** (não pelo índice 0 "decorado").
- Empacotado em **função reutilizável** que também devolve o tamanho médio das caixas (usado pelo roteador).


In [ ]:
# Carrega o modelo YOLOv11 (na 1a vez baixa os pesos)
yolo = YOLO("yolo11n.pt")
print("YOLOv11 carregado.")

def detectar_yolo(caminho_img, conf=0.5):
    '''Detecta pessoas e retorna: contagem, imagem anotada (RGB), razão média de área das caixas.'''
    res = yolo(caminho_img, conf=conf, verbose=False)[0]
    nomes = res.names
    pessoas = [b for b in res.boxes if nomes[int(b.cls)] == "person"]
    n = len(pessoas)

    # razão média (área da caixa / área da imagem): grande = pessoa perto (esparso); minúsculo = denso
    H, W = res.orig_shape
    area_img = H * W
    if n:
        razoes = []
        for b in pessoas:
            x1, y1, x2, y2 = b.xyxy[0].tolist()
            razoes.append(((x2-x1)*(y2-y1)) / area_img)
        razao_media = float(np.mean(razoes))
    else:
        razao_media = 0.0

    anotada = res.plot()[:, :, ::-1]   # BGR -> RGB (correção do bug do original)
    return n, anotada, razao_media


## 2. Modelo B — CSRNet (multidão densa, com fine-tuning)

O CSRNet não detecta indivíduos: prevê um **mapa de densidade** e conta pela **integral**.
O front-end é a **VGG16 pré-treinada (ImageNet)** — por isso o treino é **fine-tuning**.
Dataset: **ShanghaiTech**. Métrica: **MAE / RMSE**.


In [ ]:
# 2.1 Download do ShanghaiTech (Kaggle) com cache no Drive
DATA_ROOT = os.path.join(DRIVE_PROJ, "ShanghaiTech")

def achar_part(root, part):
    cand = glob.glob(os.path.join(root, "**", f"part_{part}*"), recursive=True)
    cand = [c for c in cand if os.path.isdir(c) and os.path.isdir(os.path.join(c, "train_data"))]
    return cand[0] if cand else None

if achar_part(DATA_ROOT, PART) is None:
    print("Dataset não está no Drive. Baixando do Kaggle (precisa do kaggle.json)...")
    from google.colab import files
    print("Suba o seu kaggle.json:")
    files.upload()
    os.makedirs("/root/.kaggle", exist_ok=True)
    os.replace("kaggle.json", "/root/.kaggle/kaggle.json"); os.chmod("/root/.kaggle/kaggle.json", 0o600)
    !pip install -q kaggle
    !kaggle datasets download -d tthien/shanghaitech -p /content --unzip
    src = None
    for d in glob.glob("/content/**/part_A*", recursive=True):
        src = os.path.dirname(d); break
    os.makedirs(DATA_ROOT, exist_ok=True)
    !cp -r "{src}/." "{DATA_ROOT}/"
    print("Copiado para o Drive.")

PART_DIR = achar_part(DATA_ROOT, PART)
assert PART_DIR, "Não localizei a pasta da parte escolhida."
print("Usando:", PART_DIR)


In [ ]:
# 2.2 Mapa de densidade + Dataset
def listar(split):
    img_dir = os.path.join(PART_DIR, f"{split}_data", "images")
    gt_dir  = os.path.join(PART_DIR, f"{split}_data", "ground-truth")
    pares = []
    for ip in sorted(glob.glob(os.path.join(img_dir, "*.jpg"))):
        n = os.path.splitext(os.path.basename(ip))[0]
        gp = os.path.join(gt_dir, f"GT_{n}.mat")
        if os.path.exists(gp): pares.append((ip, gp))
    return pares

def ler_pontos(matpath):
    return sio.loadmat(matpath)['image_info'][0,0][0,0][0]   # Nx2 (x,y) formato ShanghaiTech

def densidade(h, w, pts, sigma=SIGMA):
    d = np.zeros((h, w), np.float32)
    for x, y in pts:
        xi, yi = int(round(x)), int(round(y))
        if 0 <= yi < h and 0 <= xi < w: d[yi, xi] = 1.0
    return gaussian_filter(d, sigma=sigma)

NORM = transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])

class ShanghaiTech(Dataset):
    def __init__(self, split, treino):
        self.pares = listar(split)
        if treino and MAX_TRAIN: self.pares = self.pares[:MAX_TRAIN]
        self.treino = treino
    def __len__(self): return len(self.pares)
    def __getitem__(self, i):
        ip, gp = self.pares[i]
        img = cv2.cvtColor(cv2.imread(ip), cv2.COLOR_BGR2RGB)
        pts = ler_pontos(gp); h, w = img.shape[:2]
        d = densidade(h, w, pts)
        if self.treino:
            ch, cw = max((h//2)//8*8, 8), max((w//2)//8*8, 8)
            y0 = random.randint(0, h-ch); x0 = random.randint(0, w-cw)
            img = img[y0:y0+ch, x0:x0+cw]; d = d[y0:y0+ch, x0:x0+cw]
        else:
            ch, cw = h//8*8, w//8*8
            img = img[:ch, :cw]; d = d[:ch, :cw]
        alvo = cv2.resize(d, (cw//8, ch//8), interpolation=cv2.INTER_LINEAR) * 64.0  # casa com saída 1/8
        x = NORM(torch.from_numpy(img/255.).permute(2,0,1).float())
        y = torch.from_numpy(alvo).unsqueeze(0).float()
        return x, y

print("Treino:", len(ShanghaiTech("train", True)), "| Teste:", len(ShanghaiTech("test", False)))


In [ ]:
# 2.3 Modelo CSRNet (VGG16 pré-treinada + back-end dilatado)
class CSRNet(nn.Module):
    def __init__(self, pretrained=True):
        super().__init__()
        self.frontend = self._mk([64,64,'M',128,128,'M',256,256,256,'M',512,512,512], 3, 1)
        self.backend  = self._mk([512,512,512,256,128,64], 512, 2)
        self.output   = nn.Conv2d(64, 1, 1)
        if pretrained:
            vgg = models.vgg16(weights=models.VGG16_Weights.IMAGENET1K_V1)
            pre = [m for m in vgg.features if isinstance(m, nn.Conv2d)][:10]
            own = [m for m in self.frontend if isinstance(m, nn.Conv2d)]
            for o, p in zip(own, pre):
                o.weight.data.copy_(p.weight.data); o.bias.data.copy_(p.bias.data)
    def _mk(self, cfg, cin, d):
        L = []
        for v in cfg:
            if v == 'M': L += [nn.MaxPool2d(2,2)]
            else: L += [nn.Conv2d(cin, v, 3, padding=d, dilation=d), nn.ReLU(inplace=True)]; cin = v
        return nn.Sequential(*L)
    def forward(self, x):
        return self.output(self.backend(self.frontend(x)))

csrnet = CSRNet(pretrained=True).to(DEVICE)
print("CSRNet:", round(sum(p.numel() for p in csrnet.parameters())/1e6,1), "M parâmetros")


In [ ]:
# 2.4 Avaliação (MAE / RMSE) e baseline ANTES do treino
@torch.no_grad()
def avaliar(model, loader):
    model.eval(); ae = se = 0.0; n = 0
    for x, y in loader:
        pred = model(x.to(DEVICE)).sum().item(); real = y.sum().item()
        ae += abs(pred-real); se += (pred-real)**2; n += 1
    return ae/n, (se/n)**0.5

train_loader = DataLoader(ShanghaiTech("train", True), batch_size=1, shuffle=True)
test_loader  = DataLoader(ShanghaiTech("test", False), batch_size=1, shuffle=False)
mae0, rmse0 = avaliar(csrnet, test_loader)
print(f"BASELINE (antes do treino) -> MAE={mae0:.1f}  RMSE={rmse0:.1f}")


In [ ]:
# 2.5 Treino com checkpoint + retomada automática + log por época
opt = torch.optim.Adam(csrnet.parameters(), lr=LR)
loss_fn = nn.MSELoss()
LAST = os.path.join(CKPT_DIR, "last.pth"); BEST = os.path.join(CKPT_DIR, "best.pth")
CSV  = os.path.join(CKPT_DIR, "results.csv")
inicio, melhor, hist = 0, float("inf"), []

if os.path.exists(LAST):
    ck = torch.load(LAST, map_location=DEVICE)
    csrnet.load_state_dict(ck["model"]); opt.load_state_dict(ck["opt"])
    inicio, melhor, hist = ck["epoch"]+1, ck["melhor"], ck["hist"]
    print(f"Retomando da época {inicio} (melhor MAE: {melhor:.1f})")

for ep in range(inicio, EPOCHS):
    csrnet.train(); t0 = time.time(); soma = 0.0
    for x, y in train_loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        opt.zero_grad(); loss = loss_fn(csrnet(x), y); loss.backward(); opt.step()
        soma += loss.item()
    mae, rmse = avaliar(csrnet, test_loader)
    hist.append({"epoca": ep+1, "loss": soma/len(train_loader), "mae": mae, "rmse": rmse})
    print(f"época {ep+1:3d}/{EPOCHS} | loss {soma/len(train_loader):.3f} | MAE {mae:.1f} | RMSE {rmse:.1f} | {time.time()-t0:.0f}s")
    ck = {"epoch": ep, "model": csrnet.state_dict(), "opt": opt.state_dict(), "melhor": melhor, "hist": hist}
    torch.save(ck, LAST)
    if mae < melhor:
        melhor = mae; ck["melhor"] = melhor; torch.save(ck, BEST); print(f"   novo melhor MAE: {melhor:.1f}")
    with open(CSV, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=["epoca","loss","mae","rmse"]); w.writeheader(); w.writerows(hist)
print("Treino concluído. Melhor MAE:", round(melhor,1))


In [ ]:
# 2.6 Curvas de aprendizado
ep=[h["epoca"] for h in hist]; loss=[h["loss"] for h in hist]; mae=[h["mae"] for h in hist]
fig, ax = plt.subplots(1,2, figsize=(13,4))
ax[0].plot(ep, loss); ax[0].set_title("Loss de treino"); ax[0].set_xlabel("época"); ax[0].grid(alpha=.3)
ax[1].plot(ep, mae);  ax[1].set_title("MAE no teste");   ax[1].set_xlabel("época"); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.show()


In [ ]:
# 2.7 Antes x Depois (melhor checkpoint)
csrnet.load_state_dict(torch.load(BEST, map_location=DEVICE)["model"])
mae_f, rmse_f = avaliar(csrnet, test_loader)
print(f"{'':10}{'MAE':>8}{'RMSE':>9}")
print(f"{'ANTES':10}{mae0:8.1f}{rmse0:9.1f}")
print(f"{'DEPOIS':10}{mae_f:8.1f}{rmse_f:9.1f}")
print(f"{'ganho':10}{mae0-mae_f:8.1f}{rmse0-rmse_f:9.1f}")


## 3. Roteador híbrido

**O problema do ovo e da galinha:** para escolher o modelo "pela quantidade de gente", seria
preciso já saber a quantidade — que é o que queremos descobrir. **Solução:** o YOLO é confiável
*justamente quando a cena é esparsa*. Então rodamos o YOLO primeiro e usamos dois sinais baratos:
a **contagem** dele e o **tamanho médio das caixas** (cabeças pequenas = multidão). Se a cena for
densa, escalamos para o CSRNet. Os limiares são parâmetros calibráveis (definidos no início).


In [ ]:
# Inferência do CSRNet numa imagem qualquer + função de decisão
@torch.no_grad()
def inferir_csrnet(caminho_img):
    csrnet.eval()
    img = cv2.cvtColor(cv2.imread(caminho_img), cv2.COLOR_BGR2RGB)
    h, w = img.shape[0]//8*8, img.shape[1]//8*8
    img = img[:h, :w]
    x = NORM(torch.from_numpy(img/255.).permute(2,0,1).float()).unsqueeze(0).to(DEVICE)
    dmap = csrnet(x).squeeze().cpu().numpy()
    return float(dmap.sum()), dmap, img

def eh_denso(n_yolo, razao_media):
    '''Decide se a cena é densa a partir da contagem do YOLO e do tamanho médio das caixas.'''
    if n_yolo > LIMIAR_CONTAGEM:
        return True
    if n_yolo >= LIMIAR_CONTAGEM_MIN and razao_media < LIMIAR_TAMANHO:
        return True
    return False

def contar(caminho_img):
    '''Roteador: roda o YOLO, decide o regime e devolve a contagem do modelo apropriado.'''
    n_yolo, anotada, razao = detectar_yolo(caminho_img)
    if eh_denso(n_yolo, razao):
        cont, dmap, img = inferir_csrnet(caminho_img)
        return {"modo": "DENSO (CSRNet)", "contagem": round(cont),
                "yolo_bruto": n_yolo, "razao_caixa": round(razao,4),
                "viz": dmap, "img": img, "denso": True}
    return {"modo": "ESPARSO (YOLO)", "contagem": n_yolo,
            "yolo_bruto": n_yolo, "razao_caixa": round(razao,4),
            "viz": anotada, "denso": False}
print("Roteador pronto.")


## 4. Demonstração integrada

Suba uma imagem (ou use o exemplo). O sistema decide sozinho qual modelo usar e mostra o resultado.


In [ ]:
# Entrada (upload ou exemplo) + decisão + visualização
caminho = None
try:
    from google.colab import files
    print("Selecione uma imagem (ou cancele para usar o exemplo):")
    env = files.upload()
    if env:
        caminho = next(iter(env))
except Exception:
    pass
if not caminho:
    import urllib.request
    caminho = "exemplo.jpg"
    urllib.request.urlretrieve("https://ultralytics.com/images/bus.jpg", caminho)
    print("Usando imagem de exemplo.")

r = contar(caminho)
print(f"\nMODO: {r['modo']}  |  CONTAGEM: {r['contagem']}")
print(f"(sinais -> YOLO bruto: {r['yolo_bruto']}  |  tamanho médio de caixa: {r['razao_caixa']})")

plt.figure(figsize=(12,7))
if r["denso"]:
    plt.imshow(r["viz"], cmap="jet"); plt.title(f"DENSO — CSRNet estimou {r['contagem']} pessoas")
else:
    plt.imshow(r["viz"]); plt.title(f"ESPARSO — YOLO detectou {r['contagem']} pessoas")
plt.axis("off"); plt.show()


## Notas técnicas e limitações (honestas)

- **Fine-tuning:** o front-end VGG16 entra pré-treinado; o treino o adapta à contagem de cabeça.
- **Métrica:** MAE/RMSE comparam a integral do mapa previsto com a real. O alvo é gerado em 1/8 da
  resolução (saída do modelo) e ×64 para preservar a contagem.
- **Roteador:** o limiar pode errar na fronteira (cenas de ~100-150 pessoas). É um parâmetro a calibrar
  com o próprio estudo comparativo — apontar isso no relatório é ponto a favor (mostra rigor).
- **Ground truth:** kernel gaussiano fixo (σ=15). Para a Parte A, o paper usa kernel geometria-adaptativo
  (melhora um pouco o MAE) — fica como upgrade opcional.
- **GPU grátis:** treino com checkpoint por época e retomada automática. Referência de sanidade:
  CSRNet público fica em MAE ~68 (Parte A) e ~10 (Parte B); o que importa é a curva caindo.
